# 01 — Morphisms as the app: ingest → materialize → [UM]

This notebook **is the application**. Each code cell is a step of the app,
and the [UM] (`ewm-app::StateMachine`) runs them against the shared
`StateCache`.

Progression (slow, one morphism at a time):

1. **ingest** — the default padded n-gram morphism: SHA1 of the new HLLSet,
   three pointers per token, LUTs + hllsetLUT side effects;
2. **materialize** — LUT-first, ordered by default, `no_order` option;
3. **the [UM]** — the stateless driver over the shared state, cell by cell.

In [2]:
:dep ewm-app = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/crates/ewm-app" }
:dep ewm-git = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/crates/ewm-git" }
:dep context-tree = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/crates/context-tree" }
:dep hllset-morphisms = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/crates/hllset-morphisms" }
:dep hllset-contracts = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/crates/hllset-contracts" }
:dep hllset-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/crates/hllset-core" }

In [3]:
use ewm_app::{StateCache, StateMachine};
use hllset_contracts::BitAddress;
use hllset_morphisms::{
    ingest, key_scheme, materialize, materialize_no_order, Ingest, CHANNELS, CHANNEL_NAMES,
    CHANNEL_SEEDS, NG, NS, N_SEEDS, PAD,
};

println!("notebook app loaded: [UM] + morphisms");

notebook app loaded: [UM] + morphisms


---
## Morphism 1 — ingest

The default case: an ordered token collection in, the **SHA1 of the new
HLLSet** out. The side effects preserve the work — token LUTs and the
hllsetLUT with the three channel HLLSet keys.

In [4]:
let tokens: Vec<Vec<u8>> = ["the", "cat", "sat"]
    .iter()
    .map(|t| t.as_bytes().to_vec())
    .collect();

let ing = ingest(&tokens);
println!("default return — SHA1 of the new HLLSet:");
println!("  projection key: {}  (scheme = {:?})", ing.key, key_scheme(&ing.key));
for (ch, key) in ing.keys.iter().enumerate() {
    println!("  G{} key: {}  (scheme = {:?})", ch + 1, key, key_scheme(key));
}
println!("hllsetLUT: {} named entries", ing.hllset_lut.len());
for (ch, key) in ing.keys.iter().enumerate() {
    println!(
        "  {} = {}  (TH = {})",
        CHANNEL_NAMES[ch],
        key,
        ing.hllset_lut.th_named(CHANNEL_NAMES[ch], key)
    );
}

default return — SHA1 of the new HLLSet:
  projection key: h:ng:a421491d50bfa3742c15672667a0bc6d9a3ed102  (scheme = Some("ng"))
  G1 key: h:ng:c1b894b666264b47d78bc061db7e63f17d746207  (scheme = Some("ng"))
  G2 key: h:ng:1bbb1d9658640931592121398acae8627f8f03f0  (scheme = Some("ng"))
  G3 key: h:ng:b7661bd623bca71dca2464c72d43fd66296706d8  (scheme = Some("ng"))
hllsetLUT: 3 named entries
  G1 = h:ng:c1b894b666264b47d78bc061db7e63f17d746207  (TH = 1)
  G2 = h:ng:1bbb1d9658640931592121398acae8627f8f03f0  (TH = 1)
  G3 = h:ng:b7661bd623bca71dca2464c72d43fd66296706d8  (TH = 1)


()

Every real token has **three hashes pointing at it** — its 1-gram, 2-gram
and 3-gram (each channel uses its own seed). Let's check the three pointers
of `"cat"` inside `[the, cat, sat]`.

In [ ]:
fn join_gram(parts: &[&[u8]]) -> Vec<u8> {
    if parts.len() == 1 {
        return parts[0].to_vec();
    }
    let mut out = Vec::new();
    for (i, p) in parts.iter().enumerate() {
        if i > 0 {
            out.push(0u8);
        }
        out.extend_from_slice(p);
    }
    out
}

let ing = ingest(["the", "cat", "sat"]);
for ch in 0..CHANNELS {
    let seed = CHANNEL_SEEDS[ch];
    let gram: Vec<u8> = match ch {
        0 => join_gram(&[b"cat"]),
        1 => join_gram(&[b"cat", b"sat"]),
        _ => join_gram(&[b"cat", b"sat", PAD]),
    };
    let addr = BitAddress::of_token_seeded(&gram, seed);
    let atom_set = ing.sketches[ch].has_bit(addr.reg(), addr.tz());
    let fiber_has_cat = ing.luts[ch].fiber(addr.bit()).contains(&b"cat".to_vec());
    println!(
        "channel {}: atom of {:?} set = {}, LUT fiber holds cat = {}",
        ch + 1,
        String::from_utf8_lossy(&gram).replace('\0', "|"),
        atom_set,
        fiber_has_cat
    );
}

The same Gn names, the other bootstrap scheme: the **n-seed** path sets
the same G1/G2/G3 channels with seeded hashes. The HLLSet is scheme-agnostic;
the `ns` prefix on its keys says: use the n-seed LUTs — the plain set, no
order.

In [ ]:
let mut ns = Ingest::new();
ns.ingest_tokens([&b"cat"[..], &b"sat"[..]]);

println!("n-seed projection key: {}  (scheme = {:?})", ns.key(), key_scheme(&ns.key()));
for (i, k) in ns.keys().iter().enumerate() {
    println!("  G{} key: {}  (scheme = {:?})", i + 1, k, key_scheme(k));
}

// n-seed LUTs are orderless: materialize returns the plain set.
let plain = {
    let pairs: Vec<_> = (0..N_SEEDS)
        .map(|i| (ns.hllset(i), ns.lut(i)))
        .collect();
    hllset_morphisms::materialize::materialize(&pairs, ns.tf())
};
println!("n-seed materialize (set): {:?}", plain);
println!("scheme ng={} ns={}", NG, NS);

---
## Morphism 2 — materialize

LUT-first over the three channels; **ordered by default**, `no_order`
returns the plain set.

In [ ]:
let tokens: Vec<Vec<u8>> = ["the", "cat", "sat", "on", "the", "mat"]
    .iter()
    .map(|t| t.as_bytes().to_vec())
    .collect();
let ing = ingest(&tokens);

let ordered = materialize(&ing);
println!("ordered restore == original: {}", ordered == tokens);

let set = materialize_no_order(&ing);
println!("no_order set: {:?}", set);

---
## The [UM] — this notebook is the app

The [UM] (`StateMachine`) is stateless: it owns only the store handle.
S(t) and H(t-1) live in the shared `StateCache`. Each cell below is one
turn of the app.

In [ ]:
let mut um = StateMachine::new(ewm_git::MemoryStore::default());
let mut cache = StateCache::empty();

let turn1 = [10u32, 20, 30];
let out1 = um.run_turn(&mut cache, &turn1).expect("turn 1");
println!("commit: {}", out1.commit.as_ref().map(|c| c.to_string()).unwrap_or_else(|| "-".into()));
println!("tip:    {}", out1.head.as_ref().map(|h| h.to_string()).unwrap_or_else(|| "-".into()));
println!("S(t) leaves: {}", out1.tree.leaves().len());
println!("full_image: {:?}", out1.full_image);
{
    let v = out1.commit_view.as_ref().expect("root view");
    println!("D={} R={} N={}", v.departed.popcount(), v.retained.popcount(), v.new.popcount());
}

In [ ]:
let turn2 = [20u32, 30, 40];
let out2 = um.run_turn(&mut cache, &turn2).expect("turn 2");
println!("commit: {}", out2.commit.as_ref().map(|c| c.to_string()).unwrap_or_else(|| "-".into()));
{
    let v = out2.commit_view.as_ref().expect("head view");
    println!("D={} R={} N={}", v.departed.popcount(), v.retained.popcount(), v.new.popcount());
}
println!(
    "tree D/R/N: added={:?} retained={:?} removed={:?}",
    out2.diff.added, out2.diff.retained, out2.diff.removed
);

---
## Recovery — pop, not rebuild

Crash: drop the [UM] **and** its cache. The persistent store is untouched; a
fresh [UM] and a restored cache resume from the tip.

In [ ]:
let dir = std::env::temp_dir().join("um-notebook-recovery");
let _ = std::fs::remove_dir_all(&dir);

// First [UM] commits one turn, then "crashes" (goes out of scope).
{
    let mut um = StateMachine::new(ewm_git::LooseStore::new(&dir));
    let mut cache = StateCache::empty();
    um.run_turn(&mut cache, &[1u32, 2, 3]).expect("turn");
    println!("tip after first [UM]: {:?}", um.head().map(|h| h.to_string()));
}

// Fresh [UM] over the same store; fresh cache restored from the tip.
let mut um2 = StateMachine::open(ewm_git::LooseStore::new(&dir));
let mut cache2 = StateCache::restore(um2.repo());
println!("restored tip:        {:?}", um2.head().map(|h| h.to_string()));
println!("restored cache.tip:  {:?}", cache2.tip.as_ref().map(|h| h.to_string()));
println!("restored leaves:     {}", cache2.tree().leaves().len());

let resumed = um2.run_turn(&mut cache2, &[3u32, 4]).expect("resume");
println!("resumed commit:      {:?}", resumed.commit.as_ref().map(|c| c.to_string()));
let _ = std::fs::remove_dir_all(&dir);

---
## Summary

- **ingest** returns the SHA1 of the new HLLSet and preserves the work in
  LUTs + hllsetLUT;
- **materialize** restores the ordered token collection (or the plain set
  with `no_order`);
- the **[UM]** runs this notebook cell by cell as a stateless driver, with
  S(t) and H(t-1) outside it in the shareable `StateCache`;
- recovery is a fresh [UM] + `StateCache::restore` — a read of the tip, not
  a replay.